# Do Machines "Understand" Meaning Across Languages? (Colab version)

When we translate *"I like coffee"* into French (*"J'aime le café"*), the words change but the **meaning** stays the same. Can a computer learn to represent that shared meaning?

Modern language models convert every sentence into a **vector** — a list of numbers that places the sentence somewhere in a high-dimensional space. If the model has learned something about meaning, sentences that *mean the same thing* should end up **near each other**, regardless of language.

In this notebook we'll:

1. Build intuition for **cosine similarity** — the ruler we use to measure "nearness".
2. Compare two models — **mBERT** and **LaBSE** — to see whether they organise sentences by *language* or by *meaning*.
3. Reflect on what this tells us about how machines process language.

## Setup

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib

## Part 1 — What is cosine similarity?

Imagine you have two arrows pinned at the same point. Cosine similarity measures the **angle** between them, ignoring how long they are.

- If the arrows point in **exactly the same direction**, the cosine of the angle is **1** (identical).
- If they point in **completely unrelated directions** (at 90 degrees), the cosine is **0**.
- If they point in **opposite directions**, the cosine is **-1**.

In practice, sentence vectors live in hundreds of dimensions, not just two — but the idea is the same. Two sentences with a cosine similarity close to 1 are "pointing the same way" in meaning-space.

The formula is:

$$\text{cosine similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}| \times |\mathbf{b}|}$$

Don't worry about the maths — the key intuition is: **same direction = similar meaning**.

In [ ]:
# A quick visual: cosine similarity in 2D
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

cases = [
    ("Same direction (cos = 1.0)",  [2, 1],   [4, 2]),
    ("Unrelated (cos ≈ 0)",         [2, 1],   [-1, 2]),
    ("Opposite (cos = -1.0)",       [2, 1],   [-2, -1]),
]

for ax, (title, a, b) in zip(axes, cases):
    origin = [0, 0]
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    ax.annotate("", xy=a, xytext=origin,
                arrowprops=dict(arrowstyle="->", color="#2196F3", lw=2.5))
    ax.annotate("", xy=b, xytext=origin,
                arrowprops=dict(arrowstyle="->", color="#F44336", lw=2.5))
    ax.text(a[0]*1.08, a[1]*1.08, "a", color="#2196F3", fontsize=13, fontweight="bold")
    ax.text(b[0]*1.08, b[1]*1.08, "b", color="#F44336", fontsize=13, fontweight="bold")

    lim = max(abs(a[0]), abs(a[1]), abs(b[0]), abs(b[1])) + 1
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    ax.grid(alpha=0.2)
    ax.set_title(f"{title}\ncos = {cos:.2f}", fontsize=10, fontweight="bold")

fig.suptitle("Cosine similarity in 2D", fontsize=13, fontweight="bold", y=1.04)
fig.tight_layout()
plt.show()

## Part 2 — Our test sentences

We use 10 simple sentences, each translated into four languages: English, French, Spanish, and German — 40 sentences total.

The question: will *"I like coffee"* be closer to *"J'aime le café"* (same meaning, different language) or to *"The weather is nice"* (different meaning, same language)?

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

pairs = [
    (0, "I like coffee.",              "J'aime le café.",                      "Me gusta el café.",               "Ich mag Kaffee."),
    (1, "The weather is nice.",        "Il fait beau.",                        "Hace buen tiempo.",               "Das Wetter ist schön."),
    (2, "Where is the station?",       "Où est la gare ?",                     "¿Dónde está la estación?",        "Wo ist der Bahnhof?"),
    (3, "This book is interesting.",    "Ce livre est intéressant.",            "Este libro es interesante.",      "Dieses Buch ist interessant."),
    (4, "She plays the piano.",        "Elle joue du piano.",                  "Ella toca el piano.",             "Sie spielt Klavier."),
    (5, "We need more data.",          "Nous avons besoin de plus de données.","Necesitamos más datos.",          "Wir brauchen mehr Daten."),
    (6, "Close the window, please.",   "Fermez la fenêtre, s'il vous plaît.", "Cierra la ventana, por favor.",   "Bitte schließ das Fenster."),
    (7, "I lost my keys.",             "J'ai perdu mes clés.",                 "Perdí mis llaves.",               "Ich habe meine Schlüssel verloren."),
    (8, "Time is money.",              "Le temps, c'est de l'argent.",         "El tiempo es dinero.",            "Zeit ist Geld."),
    (9, "He loves to travel.",         "Il aime voyager.",                     "Le encanta viajar.",              "Er reist gern."),
]

langs = ["EN", "FR", "ES", "DE"]

items = []
for gid, en, fr, es, de in pairs:
    items.extend([(gid, "EN", en), (gid, "FR", fr), (gid, "ES", es), (gid, "DE", de)])

texts      = [t for (_, _, t) in items]
labels     = [f"{lang}-{gid}" for (gid, lang, _) in items]
lang_only  = [lang for (_, lang, _) in items]
group_ids  = [gid  for (gid, _, _) in items]
n          = len(items)

order          = sorted(range(n), key=lambda i: (group_ids[i], lang_only[i]))
labels_ordered = [labels[i] for i in order]

print(f"{n} sentences ({len(pairs)} meanings × {len(langs)} languages)")

## Part 3 — Helper functions

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """Average token embeddings, ignoring padding tokens."""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(sentences, tokenizer, model, pooling="mean", batch_size=16):
    """Return L2-normalised sentence vectors."""
    all_vecs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=128, return_tensors="pt").to(DEVICE)
        out = model(**enc)
        if pooling == "cls":
            vecs = out.last_hidden_state[:, 0]
        else:
            vecs = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(vecs.cpu())
    emb = torch.cat(all_vecs, 0).numpy()
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)

def similarity_stats(S):
    """Return avg cosine similarity for same-meaning, same-language, and neither."""
    same_meaning, same_lang, diff_both = [], [], []
    for i in range(n):
        for j in range(i + 1, n):
            sim = S[i, j]
            if group_ids[i] == group_ids[j]:
                same_meaning.append(sim)
            elif lang_only[i] == lang_only[j]:
                same_lang.append(sim)
            else:
                diff_both.append(sim)
    return {
        "Same meaning, diff language": np.mean(same_meaning),
        "Same language, diff meaning": np.mean(same_lang),
        "Different both":              np.mean(diff_both),
    }

def plot_heatmap(S, title, vmin=0.4, vmax=1.0):
    """Plot a cosine-similarity heatmap grouped by meaning."""
    S_ordered = S[np.ix_(order, order)]
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(S_ordered, cmap="RdYlBu_r", vmin=vmin, vmax=vmax, aspect="equal")
    ax.set_xticks(range(n))
    ax.set_xticklabels(labels_ordered, rotation=90, fontsize=5.5, fontfamily="monospace")
    ax.set_yticks(range(n))
    ax.set_yticklabels(labels_ordered, fontsize=5.5, fontfamily="monospace")
    for k in range(1, len(pairs)):
        pos = k * len(langs) - 0.5
        ax.axhline(pos, color="white", linewidth=1)
        ax.axvline(pos, color="white", linewidth=1)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine similarity", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    fig.tight_layout()
    plt.show()

## Part 4 — Model A: mBERT (Multilingual BERT)

**mBERT** was trained by Google in 2018. It learned to read 104 languages by predicting masked words — for example, given *"I like ___"*, it learns to guess *"coffee"*. It was **never explicitly told** that *"I like coffee"* and *"J'aime le café"* mean the same thing.

In [ ]:
print("Loading mBERT...")
tok_mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
mdl_mbert = AutoModel.from_pretrained("bert-base-multilingual-cased").to(DEVICE)
mdl_mbert.eval()

emb_mbert = encode(texts, tok_mbert, mdl_mbert, pooling="mean")
S_mbert   = cosine_similarity(emb_mbert, emb_mbert)

plot_heatmap(S_mbert, "mBERT — cosine similarity\n(rows/cols grouped by meaning)")

stats_mbert = similarity_stats(S_mbert)
for k, v in stats_mbert.items():
    print(f"  {k}: {v:.4f}")

### Reading the mBERT heatmap

You'll likely find that "same language, diff meaning" and "same meaning, diff language" are **close together** — mBERT doesn't strongly separate the two. Language identity "leaks" into the representation.

This makes sense: mBERT was never *told* that translations are equivalent.

## Part 5 — Model B: LaBSE (Language-agnostic BERT Sentence Embedding)

**LaBSE** was trained specifically to produce **language-agnostic** sentence embeddings. During training, it was shown millions of translation pairs and learned that translations should map to nearly the **same point** in vector space. It supports 109 languages.

In [ ]:
print("Loading LaBSE...")
tok_labse = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")
mdl_labse = AutoModel.from_pretrained("sentence-transformers/LaBSE").to(DEVICE)
mdl_labse.eval()

emb_labse = encode(texts, tok_labse, mdl_labse, pooling="cls")
S_labse   = cosine_similarity(emb_labse, emb_labse)

plot_heatmap(S_labse, "LaBSE — cosine similarity\n(rows/cols grouped by meaning)")

stats_labse = similarity_stats(S_labse)
for k, v in stats_labse.items():
    print(f"  {k}: {v:.4f}")

### Reading the LaBSE heatmap

The difference should be dramatic. LaBSE's "same meaning" score is much higher than "same language" — the heatmap shows crisp 4x4 blocks. The model has learned to erase surface language differences and keep only meaning.

## Part 6 — Side-by-side comparison

In [ ]:
# --- Side-by-side heatmaps ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, S, title in [(ax1, S_mbert, "mBERT"), (ax2, S_labse, "LaBSE")]:
    S_ord = S[np.ix_(order, order)]
    im = ax.imshow(S_ord, cmap="RdYlBu_r", vmin=0.4, vmax=1.0, aspect="equal")
    ax.set_xticks(range(n))
    ax.set_xticklabels(labels_ordered, rotation=90, fontsize=5, fontfamily="monospace")
    ax.set_yticks(range(n))
    ax.set_yticklabels(labels_ordered, fontsize=5, fontfamily="monospace")
    for k in range(1, len(pairs)):
        pos = k * len(langs) - 0.5
        ax.axhline(pos, color="white", linewidth=0.8)
        ax.axvline(pos, color="white", linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight="bold")

cbar = fig.colorbar(im, ax=[ax1, ax2], fraction=0.02, pad=0.02)
cbar.set_label("Cosine similarity", fontsize=11)
fig.suptitle("Do translations cluster together?",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# --- Bar chart ---
model_names = ["mBERT", "LaBSE"]
all_stats   = [stats_mbert, stats_labse]
categories  = list(all_stats[0].keys())
colours     = ["#2196F3", "#FF9800", "#9E9E9E"]

x = np.arange(len(model_names))
width = 0.22

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, (cat, col) in enumerate(zip(categories, colours)):
    vals = [s[cat] for s in all_stats]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=cat, color=col)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Average cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontweight="bold", fontsize=12)
ax.legend(fontsize=8.5, loc="upper right")
ax.set_ylim(0.3, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.set_title("Meaning vs language clustering", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## What does this tell us?

The key takeaway: **what a model learns depends on what it was trained to do**.

- **mBERT** was trained to fill in blanks within a single language. It picks up *some* cross-lingual structure, but language identity still "leaks" into its vectors.
- **LaBSE** was trained on translation pairs — it was directly rewarded for mapping translations to the same point. The result: meaning dominates, language is nearly invisible.

The same architecture (BERT), given different training objectives, can learn to either **preserve** or **erase** linguistic identity. What counts as "similar" is a consequence of what the model was *optimised for*.

---

## Exercises

### Exercise 1 — Add a distant language

Add a fifth language to the `pairs` list — choose one typologically distant from the European four (e.g., Japanese, Arabic, Swahili, Hindi). Re-run the notebook.
- Does LaBSE still cluster the new language with the correct meaning group?
- Does mBERT struggle more? Why might that be?

### Exercise 2 — Paraphrases instead of translations

Replace the four translations of one sentence with **paraphrases in the same language** (e.g., *"I like coffee"* becomes *"Coffee is my favourite drink"*). Does the model still group them together?

### Exercise 3 — Idioms and irony

Try sentences where meaning is culturally specific:
- *"It's raining cats and dogs"* vs its literal translation vs its actual equivalent (*"Il pleut des cordes"*)
- *"Oh great, another meeting"* vs *"I'm excited about the meeting"*

Which pairs does LaBSE rate as similar? Where does it fail?

### Exercise 4 — Written reflection (no code)

In 300-500 words: A model like LaBSE places *"Time is money"* and *"Zeit ist Geld"* at nearly the same point in vector space. Does this mean it "understands" the sentence? Consider Benjamin's *"The Task of the Translator"* or Jakobson's distinction between interlingual and intralingual translation — how would these thinkers respond to cosine similarity of 0.98 meaning "same meaning"?

### Reading the mBERT heatmap

You'll likely find that "same language, diff meaning" and "same meaning, diff language" are **close together** — mBERT doesn't strongly separate the two. Language identity "leaks" into the representation. This makes sense: mBERT was never *told* that translations are equivalent.

## Part 5 — Model B: LaBSE (Language-agnostic BERT Sentence Embedding)

**LaBSE** was trained specifically to produce **language-agnostic** sentence embeddings. During training, it was shown millions of translation pairs and learned that *"I like coffee"* and *"J'aime le café"* should map to nearly the **same point** in vector space. It supports 109 languages.